# **Training EfficientNet-B0 — 5-Fold Cross Validation (Google Colab)**

Notebook ini melatih **EfficientNet-B0** dengan **Stratified 5-Fold Cross Validation** di Google Colab.
Sudah dilengkapi otomatis dengan:
- **Summary Card Visual** per Optimizer (Blue Terminal Style)
- **Grafik Perbandingan Proses Training 4 Optimizer**
- **Visualisasi Deteksi Anomali Hierarkis (Stem, MBConv, & Grad-CAM)**
- **Error Analysis Collector** (Penyaringan FP & FN + CSV Log)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, datetime, json, sys, gc, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 8
EPOCHS      = 50
K_FOLDS     = 5
NUM_CLASSES = 2
CLASS_NAMES = ['AI', 'Asli']

# Cari otomatis folder dataset di Colab
possible_dirs = [
    '/content/dataset',
    '/content',
    '/content/dataset/dataset',
    '/content/drive/MyDrive/dataset',
    '/content/drive/MyDrive/KODING TA/dataset',
    '/content/drive/MyDrive/dataset/dataset'
]

possible_zips = [
    '/content/drive/MyDrive/dataset.zip',
    '/content/drive/MyDrive/KODING TA/dataset.zip',
    '/content/dataset.zip'
]

for zp in possible_zips:
    if os.path.exists(zp) and not any(os.path.exists(os.path.join(d, 'train_val')) for d in possible_dirs):
        print(f'[INFO] Mengekstrak {zp} ke /content...')
        import zipfile
        with zipfile.ZipFile(zp, 'r') as zip_ref:
            zip_ref.extractall('/content')

BASE_DIR = None
for d in possible_dirs:
    if os.path.exists(os.path.join(d, 'train_val')):
        BASE_DIR = d
        break

if not BASE_DIR:
    for root, dirs, files in os.walk('/content'):
        if 'train_val' in dirs:
            BASE_DIR = root
            break

if BASE_DIR:
    TRAIN_VAL_DIR = os.path.join(BASE_DIR, 'train_val')
    TEST_DIR      = os.path.join(BASE_DIR, 'test')
    print(f'[OK] Dataset ditemukan di: {BASE_DIR}')
else:
    print("[ERROR] Folder dataset/train_val tidak ditemukan!")
    print("Pastikan Anda sudah meng-upload dataset.zip ke Google Drive dan mengklik Run sel Mount Google Drive terlebih dahulu.")
    TRAIN_VAL_DIR = '/content/dataset/train_val'
    TEST_DIR      = '/content/dataset/test'

DRIVE_SAVE_DIR = f'/content/drive/MyDrive/HASIL_TRAINING_B0_KFOLD'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print('=' * 60)
print('  GOOGLE COLAB — EFFICIENTNET-B0 (5-FOLD K-FOLD)')
print('=' * 60)
print(f'  Dataset Path : {BASE_DIR}')
print(f'  Drive Output : {DRIVE_SAVE_DIR}')
print('=' * 60)

In [ ]:
def get_optimizer(opt_name):
    if opt_name == 'Baseline (RMSprop)': return tf.keras.optimizers.RMSprop(learning_rate=1e-4)
    elif opt_name == 'Adam': return tf.keras.optimizers.Adam(learning_rate=1e-4)
    elif opt_name == 'SGD': return tf.keras.optimizers.SGD(learning_rate=1e-3, momentum=0.9)
    elif opt_name == 'AdamW':
        if hasattr(tf.keras.optimizers, 'AdamW'): return tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4)
        elif hasattr(tf.keras.optimizers, 'experimental') and hasattr(tf.keras.optimizers.experimental, 'AdamW'): return tf.keras.optimizers.experimental.AdamW(learning_rate=1e-4, weight_decay=1e-4)
        else: return tf.keras.optimizers.Adam(learning_rate=1e-4)
    else: return tf.keras.optimizers.Adam(learning_rate=1e-4)

def build_model(opt_name):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = True
    for layer in base_model.layers[:-30]: layer.trainable = False
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-3))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)
    model = models.Model(inputs=base_model.input, outputs=outputs)
    model.compile(optimizer=get_optimizer(opt_name), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
filepaths, labels = [], []
for c in CLASS_NAMES:
    cdir = os.path.join(TRAIN_VAL_DIR, c)
    for f in os.listdir(cdir):
        filepaths.append(os.path.join(cdir, f))
        labels.append(c)

df_train_val = pd.DataFrame({'filepath': filepaths, 'label': labels})
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
df_train_val['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(skf.split(df_train_val, df_train_val['label'])):
    df_train_val.loc[val_idx, 'fold'] = fold + 1

print('[OK] DataFrame 5-Fold Stratified Cross Validation (Colab):')
print(df_train_val['fold'].value_counts().sort_index())

In [ ]:
# Pilihan Optimizer (Bisa Diganti di Dropdown Form Colab)
OPTIMIZER = 'AdamW' #@param ['Baseline (RMSprop)', 'Adam', 'SGD', 'AdamW']
opt_suffix = {'Baseline (RMSprop)': 'baseline', 'Adam': 'adam', 'SGD': 'sgd', 'AdamW': 'adamw'}[OPTIMIZER]

print('=' * 70)
print(f'  TRAINING COLAB: {OPTIMIZER} (EfficientNet-B0)')
print('=' * 70)

fold_metrics = []
for fold_num in range(1, K_FOLDS + 1):
    print(f'\n[{OPTIMIZER}] Fold {fold_num}/{K_FOLDS}')
    train_sub = df_train_val[df_train_val['fold'] != fold_num].reset_index(drop=True)
    val_sub   = df_train_val[df_train_val['fold'] == fold_num].reset_index(drop=True)

    train_datagen = ImageDataGenerator(
        rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
        shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest')
    val_datagen = ImageDataGenerator()

    train_gen = train_datagen.flow_from_dataframe(
        train_sub, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, shuffle=True)
    val_gen = val_datagen.flow_from_dataframe(
        val_sub, x_col='filepath', y_col='label', target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', classes=CLASS_NAMES, shuffle=False)

    tf.keras.backend.clear_session()
    gc.collect()
    model = build_model(OPTIMIZER)

    fold_weight_path = f'efficientnetb0_{opt_suffix}_fold{fold_num}.weights.h5'
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(filepath=fold_weight_path, monitor='val_accuracy', save_best_only=True, save_weights_only=True, verbose=1),
    ]

    fold_start = time.time()
    history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks, verbose=1)
    fold_elapsed = time.time() - fold_start

    val_gen.reset()
    y_pred = np.argmax(model.predict(val_gen, verbose=0), axis=1)
    y_true = val_gen.classes

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred,    average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred,        average='weighted', zero_division=0)

    fold_metrics.append({
        'fold': fold_num, 'accuracy': float(acc), 'precision': float(prec),
        'recall': float(rec), 'f1_score': float(f1),
        'epochs_run': len(history.history['loss']),
        'training_time_sec': float(fold_elapsed),
    })

    # Backup weight & history & CSV log ke Google Drive (Subfolder per Optimizer)
    try:
        OPT_DRIVE_DIR = os.path.join(DRIVE_SAVE_DIR, opt_suffix)
        os.makedirs(OPT_DRIVE_DIR, exist_ok=True)
        
        shutil.copy2(fold_weight_path, os.path.join(OPT_DRIVE_DIR, fold_weight_path))
        
        hist_path = f'history_{opt_suffix}_fold{fold_num}.json'
        with open(hist_path, 'w') as f: json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f)
        shutil.copy2(hist_path, os.path.join(OPT_DRIVE_DIR, hist_path))
        
        csv_path = f'training_log_{opt_suffix}_fold{fold_num}.csv'
        pd.DataFrame(history.history).to_csv(csv_path, index=False)
        shutil.copy2(csv_path, os.path.join(OPT_DRIVE_DIR, csv_path))
    except Exception as e:
        print(f"[WARN] Gagal backup ke Drive: {e}")

accs = [m['accuracy']*100 for m in fold_metrics]
summary = {
    'optimizer': OPTIMIZER, 'k_folds': K_FOLDS, 'folds': fold_metrics,
    'mean_accuracy': float(np.mean(accs)), 'std_accuracy': float(np.std(accs)),
}
with open(f'kfold_summary_{opt_suffix}.json', 'w') as f: json.dump(summary, f, indent=2)
try:
    OPT_DRIVE_DIR = os.path.join(DRIVE_SAVE_DIR, opt_suffix)
    os.makedirs(OPT_DRIVE_DIR, exist_ok=True)
    shutil.copy2(f'kfold_summary_{opt_suffix}.json', os.path.join(OPT_DRIVE_DIR, f'kfold_summary_{opt_suffix}.json'))
except: pass

print(f'\n[OK] Selesai training {OPTIMIZER}! Acc: {np.mean(accs):.2f}% +- {np.std(accs):.2f}%')
print(f'[OK] Seluruh file output tersimpan rapi di subfolder Drive: {os.path.join(DRIVE_SAVE_DIR, opt_suffix)}')

# ── Auto Disconnect Runtime (Penghemat Token Colab) ─────────────────
try:
    from google.colab import runtime
    print('\n[INFO] Memutus runtime GPU Colab secara otomatis untuk menghemat kuota...')
    runtime.unassign()
except Exception as e:
    print(f'\n[INFO] Training selesai. ({e})')

### **Langkah 6: Evaluasi Confusion Matrix, Summary Card & Visualisasi (Google Colab)**

In [ ]:
# Generate Confusion Matrix & Summary Card di Colab
import cv2
from sklearn.metrics import confusion_matrix

OPT_DRIVE_DIR = os.path.join(DRIVE_SAVE_DIR, opt_suffix)
os.makedirs(OPT_DRIVE_DIR, exist_ok=True)

# 1. Confusion Matrix (Best Fold)
best_fold = 1
if fold_metrics:
    best_fold = max(fold_metrics, key=lambda x: x.get('accuracy', 0))['fold']

best_weight_path = f'efficientnetb0_{opt_suffix}_fold{best_fold}.weights.h5'
if os.path.exists(best_weight_path):
    tf.keras.backend.clear_session()
    model = build_model(OPTIMIZER)
    model.load_weights(best_weight_path, by_name=True, skip_mismatch=True)
    
    test_datagen = ImageDataGenerator()
    test_gen = test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', classes=CLASS_NAMES, shuffle=False)
    y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
    y_true = test_gen.classes
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5, linecolor='gray')
    plt.title(f'Confusion Matrix — EfficientNet-B0 + {OPTIMIZER}\n(Best Fold: {best_fold})', fontsize=13, fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
    cm_path = f'confusion_matrix_{opt_suffix}_kfold.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.show()
    try: shutil.copy2(cm_path, os.path.join(OPT_DRIVE_DIR, cm_path))
    except: pass

# 2. Summary Card Visual (Blue Terminal Style)
def generate_summary_card_colab(opt_name, opt_key, test_acc_mean, test_acc_std, best_fold_num, save_path):
    fig = plt.figure(figsize=(9, 10), facecolor='#0f172a')
    ax = fig.add_subplot(111); ax.set_facecolor('#0f172a'); ax.axis('off')
    text = f"""=================================================================
 RINGKASAN HASIL {opt_name.upper()} MODEL (5-FOLD CV)
=================================================================
 Model Arsitektur : EfficientNet-B0 (Pretrained ImageNet)
 Optimizer        : {opt_name}
 Input Size       : {IMG_SIZE}
 Batch Size       : {BATCH_SIZE} | K-Folds: {K_FOLDS} Fold
 Early Stopping   : Patience 5
-----------------------------------------------------------------
 Test Accuracy    : {test_acc_mean:.2f}% +- {test_acc_std:.2f}%
 Best Fold        : Fold {best_fold_num}
================================================================="""
    ax.text(0.05, 0.95, text, transform=ax.transAxes, color='#38bdf8', fontsize=10, fontfamily='monospace', verticalalignment='top')
    plt.tight_layout(); plt.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='#0f172a'); plt.show()
    try: shutil.copy2(save_path, os.path.join(OPT_DRIVE_DIR, save_path))
    except: pass

card_name = f'summary_card_{opt_suffix}_kfold.png'
generate_summary_card_colab(OPTIMIZER, opt_suffix, np.mean(accs), np.std(accs), best_fold, card_name)
print(f"[OK] Summary Card & Confusion Matrix tersimpan di Drive: {OPT_DRIVE_DIR}")

In [ ]:
# ====================================================================
# OTOMATIS MEMUTUS RUNTIME (PENGHEMAT TOKEN)
# ====================================================================
from google.colab import runtime
print("[INFO] Memutus runtime untuk menghemat kuota...")
runtime.unassign()